# Type Reliability Check
- LLM이 만든 Label과 SBERT를 이용해서 만든 거 비교
- 둘이 차이가 얼마나 나는지 확인해야 함
- 최종 insight.csv 파일은 로컬에 있음. validation check ...

In [38]:
import pandas as pd

df3=pd.read_csv('/home/dslab/choi/Journal/LLM/Data/Academia_classified_reclassified.csv')
df3.to_csv('/home/dslab/choi/Journal/LLM/Data/Academia_reclassified_encoding.csv',index=False,encoding='utf-8-sig')

# Type Mapping
- /home/dslab/choi/Journal/LDA_ALL/Data/corpus_with_topics_academia_lda_bow.csv
- 이 데이터를 불러와서 classification별 topic 분포를 알고 싶음.
- 원하는 출력물이 type 1,2,3,4,5 별로 Topics의 %가 나와야 함.

In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 그래프 스타일 설정
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
sns.set_style('whitegrid')

print("라이브러리 로드 완료!")

라이브러리 로드 완료!


In [29]:
file_path='/home/dslab/choi/Journal/LLM/Data/Academia_classified_reclassified.csv'
# 데이터 로드
df = pd.read_csv(file_path)

print(f"데이터 크기: {df.shape}")
print(f"\n컬럼 목록:\n{df.columns.tolist()}")

데이터 크기: (4800, 16)

컬럼 목록:
['date', 'title', 'abstract', 'keywords', 'affiliations', 'english_summary', 'korean_summary', 'classification', 'evidence', 'rationale', 'vote_distribution', 'all_classifications', 'text', 'processed_tokens', 'topic', 'topic_probability']


In [30]:
# 첫 5행 확인
df.head()

,date,title,abstract,keywords,affiliations,english_summary,korean_summary,classification,evidence,rationale,vote_distribution,all_classifications,text,processed_tokens,topic,topic_probability
0,2023,Impact of content ideology on social media opi...,We offer theory and evidence regarding the imp...,"Social media, Opinion polarization, Sentiment ...",DSS,This research investigates the influence of co...,이 연구는 소셜 미디어에서 콘텐츠 이데올로기가 의견 양극화에 미치는 영향을 조사합니...,Type 2,"[""the impact of content ideology on opinion po...",This abstract fits Type 2 as it applies existi...,"{""Type 2"": 5}","[""Type 2"", ""Type 2"", ""Type 2"", ""Type 2"", ""Type...",Impact of content ideology on social media opi...,"['impact', 'ideology', 'medium', 'opinion', 'p...",2,0.739275
1,2023,A novel label-based multimodal topic model for...,Extracting useful knowledge from multimodal da...,"Multimodal data, Topic modeling, Label data, S...",DSS,This research aims to enhance knowledge extrac...,이 연구는 추천 시스템 및 교차 모달 검색과 같은 멀티미디어 애플리케이션에서 멀티모...,Type 1,"[""we propose a label-based multimodal topic (L...",This abstract fits Type 1 as it focuses on the...,NaN,NaN,A novel label-based multimodal topic model for...,"['label', 'multimodal', 'topic', 'model', 'med...",0,0.475959
2,2023,CATCHM: A novel network-based credit card frau...,Advanced fraud detection systems leverage the ...,"Network representation learning, DeepWalk, Cre...",DSS,This research aims to enhance credit card frau...,이 연구는 거래 데이터를 분석하기 위해 표현 학습을 활용한 네트워크 기반의 신용 카...,Type 1,"[""propose CATCHM, a novel network-based credit...",This abstract fits Type 1 as it focuses on the...,"{""Type 1"": 5}","[""Type 1"", ""Type 1"", ""Type 1"", ""Type 1"", ""Type...",CATCHM: A novel network-based credit card frau...,"['novel', 'network', 'credit', 'card', 'fraud_...",2,0.296221
3,2023,Exploring the effects of relationship quality ...,Although previous studies indicate the critica...,"Relationship commitment, Institution-based tru...",DSS,This study aims to explore the relationship be...,이 연구는 공급망 맥락에서 협업 상거래(c-commerce) 행동과 동적 역량 간의...,Type 2,"[""empirically examines the effects of relation...",This abstract fits Type 2 as it applies existi...,NaN,NaN,Exploring the effects of relationship quality ...,"['effect', 'relationship', 'quality', 'commerc...",1,0.329882
4,2023,Pay-for-performance schemes and hospital HIT a...,Pay-for-performance (P4P) schemes are implemen...,"Health information technology, Spillover effec...",DSS,This research investigates the dual impact of ...,이 연구는 병원에서 건강 정보 기술(HIT) 채택에 대한 성과 기반 지급(P4P) ...,Type 4,"[""Pay-for-performance (P4P) schemes are implem...",The abstract focuses primarily on the social d...,NaN,NaN,Pay-for-performance schemes and hospital HIT a...,"['performance', 'scheme', 'adoption', 'perform...",9,0.403982


In [31]:
# 데이터 정보
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4800 entries, 0 to 4799
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   date                 4800 non-null   int64  
 1   title                4800 non-null   object 
 2   abstract             4800 non-null   object 
 3   keywords             4150 non-null   object 
 4   affiliations         4800 non-null   object 
 5   english_summary      4800 non-null   object 
 6   korean_summary       4800 non-null   object 
 7   classification       4800 non-null   object 
 8   evidence             4800 non-null   object 
 9   rationale            4800 non-null   object 
 10  vote_distribution    4792 non-null   object 
 11  all_classifications  4792 non-null   object 
 12  text                 4800 non-null   object 
 13  processed_tokens     4800 non-null   object 
 14  topic                4800 non-null   int64  
 15  topic_probability    4800 non-null   f

# 4. Classification과 Topic 컬럼 식별

In [32]:
# Classification 컬럼 찾기
class_col = None
for col in df.columns:
    if 'classification' in col.lower() or 'type' in col.lower():
        class_col = col
        break

# Topic 컬럼 찾기
topic_col = None
for col in df.columns:
    if 'topic' in col.lower() or 'dominant' in col.lower():
        topic_col = col
        break

print(f"Classification 컬럼: {class_col}")
print(f"Topic 컬럼: {topic_col}")

if class_col is None or topic_col is None:
    print("\n⚠️ 경고: 컬럼을 자동으로 찾을 수 없습니다.")
    print("아래에서 수동으로 지정해주세요:")
    print(f"사용 가능한 컬럼: {df.columns.tolist()}")

Classification 컬럼: classification
Topic 컬럼: topic


In [33]:
# 필요시 수동으로 컬럼 지정 (위에서 자동으로 찾지 못한 경우)
# class_col = 'your_classification_column_name'
# topic_col = 'your_topic_column_name'

# 5. 기본 통계

In [34]:
# Type별 문서 수
print("Type별 문서 수:")
print(df[class_col].value_counts().sort_index())
print(f"\n총 문서 수: {len(df)}")

Type별 문서 수:
classification
Type 1     982
Type 2    1832
Type 3     325
Type 4    1643
Type 5      18
Name: count, dtype: int64

총 문서 수: 4800


In [35]:
# Topic별 문서 수
print("Topic별 문서 수:")
print(df[topic_col].value_counts().sort_index())

Topic별 문서 수:
topic
0     539
1     348
2     471
3     124
4     427
5     405
6     328
7     229
8     597
9     590
10    742
Name: count, dtype: int64


# 6. Type별 Topic 분포 계산

In [36]:
# Crosstab으로 빈도 계산
cross_tab = pd.crosstab(df[topic_col],df[class_col])

print("Type별 Topic 빈도:")
cross_tab

Type별 Topic 빈도:


classification,Type 1,Type 2,Type 3,Type 4,Type 5
topic,,,,,
0,312,175,20,30,2
1,38,262,5,41,2
2,34,253,13,171,0
3,18,52,5,47,2
4,85,168,79,94,1
5,27,116,30,229,3
6,18,155,25,129,1
7,23,137,14,54,1
8,68,235,27,267,0


In [37]:
# 백분율로 변환 (각 Type별로 100%가 되도록)
cross_tab_pct = cross_tab.div(cross_tab.sum(axis=1), axis=0) * 100

print("Type별 Topic 분포 (%)")
cross_tab_pct.round(2)

Type별 Topic 분포 (%)


classification,Type 1,Type 2,Type 3,Type 4,Type 5
topic,,,,,
0,57.88,32.47,3.71,5.57,0.37
1,10.92,75.29,1.44,11.78,0.57
2,7.22,53.72,2.76,36.31,0.00
3,14.52,41.94,4.03,37.90,1.61
4,19.91,39.34,18.50,22.01,0.23
5,6.67,28.64,7.41,56.54,0.74
6,5.49,47.26,7.62,39.33,0.30
7,10.04,59.83,6.11,23.58,0.44
8,11.39,39.36,4.52,44.72,0.00


# 7. 결과 저장

In [ ]:
# # CSV 파일로 저장
# output_file = 'type_topic_distribution.csv'
# cross_tab_pct.round(2).to_csv(output_file)
# print(f"결과 저장 완료: {output_file}")

# 8. 시각화
## 8.1. Heatmap-전체 분포 한눈에 보기

In [ ]:
plt.figure(figsize=(14, 6))
sns.heatmap(cross_tab_pct, annot=True, fmt='.1f', cmap='YlOrRd',
            cbar_kws={'label': 'Percentage (%)'}, linewidths=0.5)
plt.title('Topic Distribution by Classification Type (%)', fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Topic', fontsize=12)
plt.ylabel('Classification Type', fontsize=12)
plt.tight_layout()
plt.savefig('topic_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("Heatmap 저장 완료: topic_heatmap.png")

## 8.2. stacked bar chart - topic 별 type 구성

In [ ]:
plt.figure(figsize=(14, 6))
cross_tab_pct.T.plot(kind='bar', stacked=True, ax=plt.gca(),
                      colormap='Set2', width=0.8, edgecolor='black', linewidth=0.5)
plt.title('Topic Distribution by Classification Type (Stacked)', fontsize=14, fontweight='bold')
plt.xlabel('Topic', fontsize=12)
plt.ylabel('Percentage (%)', fontsize=12)
plt.legend(title='Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('topic_stacked_bar.png', dpi=300, bbox_inches='tight')
plt.show()

print("Stacked Bar Chart 저장 완료: topic_stacked_bar.png")

## 8.3. Grouped bar chart - type 별 topic 비교

In [ ]:
plt.figure(figsize=(14, 6))
cross_tab_pct.plot(kind='bar', ax=plt.gca(), width=0.8,
                   colormap='Set3', edgecolor='black', linewidth=0.5)
plt.title('Topic Distribution by Classification Type (Grouped)', fontsize=14, fontweight='bold')
plt.xlabel('Classification Type', fontsize=12)
plt.ylabel('Percentage (%)', fontsize=12)
plt.legend(title='Topic', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('topic_grouped_bar.png', dpi=300, bbox_inches='tight')
plt.show()

print("Grouped Bar Chart 저장 완료: topic_grouped_bar.png")

## 8.4. type 별 분포 - 개별 파이 차트

In [ ]:
# Type 개수 확인
n_types = len(cross_tab_pct)

# 서브플롯 생성
fig, axes = plt.subplots(1, n_types, figsize=(5*n_types, 5))

if n_types == 1:
    axes = [axes]

for idx, (type_name, row) in enumerate(cross_tab_pct.iterrows()):
    axes[idx].pie(row.values, labels=row.index, autopct='%1.1f%%', startangle=90)
    axes[idx].set_title(f'Type {type_name}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('topic_pie_charts.png', dpi=300, bbox_inches='tight')
plt.show()

print("파이 차트 저장 완료: topic_pie_charts.png")

## 9. 상세 분석 - type 별 주요 Topic

In [ ]:
print("각 Type별 주요 Topic (상위 3개):\n")
print("="*60)

for type_name, row in cross_tab_pct.iterrows():
    top_topics = row.sort_values(ascending=False).head(3)
    print(f"\nType {type_name}:")
    for topic, pct in top_topics.items():
        print(f"  - Topic {topic}: {pct:.2f}%")

## 10. 요약 통계

In [ ]:
# 각 Type의 Topic 다양성 (엔트로피)
from scipy.stats import entropy

print("Type별 Topic 다양성 (엔트로피):")
print("="*60)

for type_name, row in cross_tab_pct.iterrows():
    # 백분율을 확률로 변환
    probs = row.values / 100
    ent = entropy(probs)
    print(f"Type {type_name}: {ent:.3f}")

print("\n* 높을수록 Topic이 고르게 분포되어 있음")

## 11. 최종 결과 테이블

In [ ]:
# 스타일이 적용된 최종 테이블
styled_table = cross_tab_pct.round(2).style.background_gradient(cmap='YlOrRd', axis=1)\
    .format("{:.2f}%")\
    .set_caption("Type별 Topic 분포 (%)")

styled_table

## 분석 완료!

생성된 파일:
- `type_topic_distribution.csv`: 분포 데이터
- `topic_heatmap.png`: 히트맵
- `topic_stacked_bar.png`: 스택 막대 그래프
- `topic_grouped_bar.png`: 그룹 막대 그래프
- `topic_pie_charts.png`: 파이 차트